In [ ]:
import torch
from torch.func import jvp

from rlaopt.expression import Variable
from rlaopt.ext_tensordict import TensorDict

In [ ]:
x = Variable(torch.ones(3, 2))
y = Variable(torch.zeros(3, 2))

In [ ]:
lin_comb = 3 * x + 2 * y
lin_comb.is_affine()

In [ ]:
lin_comb.variable_values

In [ ]:
# flattened = lin_comb.variable_values.flatten(start_dim=0, end_dim=1)

In [ ]:
# Flatten each tensor
vars = lin_comb.variable_values
original_shapes = {key: vars[key].shape for key in vars.keys()}

flat_td = vars.apply(lambda x: x.flatten())
# To unflatten, you need the original shapes stored
unflat_td = TensorDict(
    {key: flat_td[key].reshape(original_shapes[key]) for key in flat_td.keys()},
    batch_size=vars.batch_size,
)

In [ ]:
vars

In [ ]:
unflat_td

In [ ]:
def flatten_td(td):
    # Flatten each tensor
    flat_td = td.apply(lambda x: x.flatten())
    original_shapes = {key: td[key].shape for key in td.keys()}
    return flat_td, original_shapes


def unflatten_td(flat_td, original_shapes):
    # To unflatten, you need the original shapes stored
    unflat_td = TensorDict(
        {key: flat_td[key].reshape(original_shapes[key]) for key in flat_td.keys()},
        batch_size=flat_td.batch_size,
    )
    return unflat_td

In [ ]:
var_values_flattened, original_shapes = flatten_td(lin_comb.variable_values)


def evaluate_lin_comb(var_values_flattened):
    var_values = unflatten_td(var_values_flattened, original_shapes)
    return lin_comb.evaluate(var_values)


output, jvp_out = jvp(
    evaluate_lin_comb, (var_values_flattened,), (torch.ones_like(var_values_flattened),)
)

In [ ]:
output

In [ ]:
jvp_out

In [ ]:
# import torch
# from tensordict import TensorDict
# from torch.func import jvp


# def model_func(input_td):
#     # A simple function that operates on a TensorDict
#     # It reads 'a' and 'b', performs a calculation, and returns a new TensorDict
#     output = 2 * input_td["a"] + 3 * input_td["b"]
#     return TensorDict({"output_key": output}, batch_size=input_td.batch_size)


# # 1. Define the primal inputs (the point at which to evaluate the Jacobian)
# primals = TensorDict(
#     {"a": torch.rand(2), "b": torch.rand(2)},
#     batch_size=[2]
# )

# # 2. Define the tangent vectors (the 'v' in JVP)
# # Tangents must have the same structure and size as the primals
# tangents = TensorDict(
#     {"a": torch.ones(2), "b": torch.ones(2)},
#     batch_size=[2]
# )

# # 3. Compute the JVP
# # jvp returns a tuple of (output_of_func, jvp_result)
# output_td, jvp_td = jvp(model_func, (primals,), (tangents,))

# print("Output TensorDict:")
# print(output_td)
# print("\nJVP TensorDict:")
# print(jvp_td)